# IndicBERT / MuRIL — Hindi + Marathi Food Safety NLP
Fine-tunes IndicBERT (free HuggingFace) to understand food safety queries in Hindi and Marathi **natively**.
Not translation — actual multilingual understanding.

In [ ]:
# pip install transformers datasets sentencepiece torch

from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import torch

# MuRIL: Multilingual Representations for Indian Languages
# Trained on Wikipedia + news in 17 Indian languages — FREE on HuggingFace
MODEL_NAME = 'google/muril-base-cased'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('Tokenizer loaded. Vocab size:', tokenizer.vocab_size)

In [ ]:
# Training data — food safety queries in EN + HI + MR
# Labels: 0=not_food_safety, 1=adulteration_query, 2=symptom_query, 3=brand_query

TRAINING_DATA = [
    # English
    ('Is turmeric powder safe to eat?',                       1),
    ('How to test milk for adulteration at home?',            1),
    ('Which honey brand is FSSAI certified?',                 3),
    ('I have stomach pain after eating food',                 2),
    ('What is the recipe for dal makhani?',                   0),
    # Hindi
    ('क्या हल्दी में मिलावट है?',                             1),
    ('दूध की जांच घर पर कैसे करें?',                         1),
    ('अमूल दूध सुरक्षित है क्या?',                           3),
    ('पेट में दर्द हो रहा है खाना खाने के बाद',              2),
    ('दाल मखनी की रेसिपी बताएं',                             0),
    # Marathi
    ('हळदीमध्ये भेसळ आहे का?',                               1),
    ('दुधाची घरी तपासणी कशी करावी?',                         1),
    ('कोणता तेलाचा ब्रँड सुरक्षित आहे?',                     3),
    ('जेवणानंतर पोटदुखी होत आहे',                            2),
    ('मसाल्यांमध्ये कोणती भेसळ असते?',                       1),
]

texts  = [t for t, _ in TRAINING_DATA]
labels = [l for _, l in TRAINING_DATA]
print(f'Training samples: {len(texts)}')

In [ ]:
# Tokenize and create dataset
def tokenize(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

dataset = Dataset.from_dict({'text': texts, 'label': labels})
dataset = dataset.train_test_split(test_size=0.2, seed=42)
tokenized = dataset.map(tokenize, batched=True)
print('Dataset ready:', tokenized)

In [ ]:
# Load model for classification
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,  # 0=other, 1=adulteration, 2=symptom, 3=brand
    id2label={0:'other', 1:'adulteration', 2:'symptom', 3:'brand'},
    label2id={'other':0, 'adulteration':1, 'symptom':2, 'brand':3},
)

args = TrainingArguments(
    output_dir='../models/muril_foodsafe',
    num_train_epochs=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    logging_steps=10,
)

import numpy as np
from datasets import load_metric

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': (preds == labels).mean()}

trainer = Trainer(
    model=model, args=args,
    train_dataset=tokenized['train'],
    eval_dataset=tokenized['test'],
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
# Test inference
def classify_query(text: str) -> str:
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
    pred = outputs.logits.argmax().item()
    labels = {0:'other', 1:'adulteration', 2:'symptom', 3:'brand'}
    return labels[pred]

tests = [
    'हल्दी में मिलावट कैसे पहचानें?',
    'दूध की जांच घर पर',
    'हळदीमध्ये भेसळ आहे का?',
    'Which milk brand is safe in Nagpur?',
]
for t in tests:
    print(f'{t[:40]:40s} → {classify_query(t)}')